In [1]:
%cd /home/dani/projects/neuroscience

/home/dani/projects/neuroscience


In [2]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np

import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

%matplotlib inline
import matplotlib.pyplot as plt

from dataclasses import dataclass
import random
import torchsde

In [3]:
b_size, state_size, brownian_size = 32, 10, 2
t_size = 20

In [4]:
def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [5]:
data = []
for i in range(30):
    data.append(np.load(f"data_processed/adhd/timeseries_{i}.npy"))

In [6]:
def pad_and_mask(batch_ts, batch_y, eps: float = 1e-5):
    """
    Pads variable-length {ts, y} to the longest T in the batch,
    ensuring ts_pad is strictly increasing.
    """
    B = len(batch_ts)
    device = batch_y[0].device
    d = batch_y[0].shape[-1]
    lengths = [t.shape[0] for t in batch_ts]
    T_max = max(lengths)

    ts_pad_list = []
    for ts in batch_ts:
        T = len(ts)
        if T < T_max:
            # create strictly increasing padded times after the last valid ts
            Δ = eps * torch.arange(1, T_max - T + 1, device=ts.device, dtype=ts.dtype)
            ts_ext = torch.cat([ts, ts[-1:] + Δ], dim=0)
        else:
            ts_ext = ts
        ts_pad_list.append(ts_ext)

    # now all have length T_max and strictly increasing
    # we can safely take one of them (e.g., the max elementwise) or just ts_pad_list[0]
    ts_pad = torch.stack(ts_pad_list, dim=0).max(dim=0).values  # (T_max,)

    # Pad y with last value; create mask
    y_pad = torch.zeros(B, T_max, d, device=device)
    mask = torch.zeros(B, T_max, device=device)
    for i, (y, L) in enumerate(zip(batch_y, lengths)):
        y_pad[i, :L] = y
        if L < T_max:
            y_pad[i, L:] = y[L - 1]  # hold-last-value padding
        mask[i, :L] = 1.0

    return ts_pad, y_pad, mask

# ----------------------------
# Synthetic dataset (variable-length)
# ----------------------------

class VariableLengthSDEData(Dataset):
    """
    Generate 1D variable-length trajectories from a *ground-truth* SDE (OU-like),
    then we'll fit a neural SDE to them.

    dy = theta*(mu - y) dt + sigma dW_t
    """
    def __init__(self,
                 n_paths: int = 512,
                 dt: float = 0.01,
                 state_size: int = 10,
                 length_range: tuple[int, int] = (10, 60),
                 theta: float = 2.0,
                 mu: float = 0.0,
                 sigma: float = 0.5,
                 device: str = "cpu"):
        self.n_paths = n_paths
        self.dt = dt
        self.state_size = state_size
        self.length_range = length_range
        self.theta = theta
        self.mu = mu
        self.sigma = sigma
        self.device = torch.device(device)
        self._paths = [self._sample_path() for _ in range(n_paths)]

    def _sample_path(self):
        T = random.randint(*self.length_range)
        ts = self.dt*torch.arange(T, device=self.device)
        #ts = torch.linspace(self.t_min, self.t_max, T, device=self.device)
        
        y = torch.zeros(T, self.state_size, device=self.device)

        # Random initial state
        y0 = torch.randn(self.state_size, device=self.device) * 0.5
        y[0] = y0

        # Euler–Maruyama to produce data
        for t in range(1, T):
            drift = self.theta * (self.mu - y[t-1])
            diffusion = self.sigma * torch.randn(self.state_size, device=self.device) / np.sqrt(1.0 / self.dt)
            # Note: For synthetic generation we keep it simple:
            y[t] = y[t-1] + drift * self.dt + self.sigma * np.sqrt(self.dt) * torch.randn_like(y[t-1])
        return ts, y

    def __len__(self):
        return self.n_paths

    def __getitem__(self, idx):
        return self._paths[idx]


def collate_variable(batch):
    # batch: list of (ts, y)
    ts_list = [b[0] for b in batch]
    y_list = [b[1] for b in batch]
    device = y_list[0].device
    ts_pad, y_pad, mask = pad_and_mask(ts_list, y_list)
    # x0 (initial states) from first timepoint of each sample
    x0 = torch.stack([y[0] for y in y_list], dim=0)  # (B, d)
    return ts_pad, y_pad, mask, x0

In [7]:
class LinearSDE(nn.Module):
    def __init__(self, state_size: int, brownian_size: int):
        super().__init__()
        self.A = nn.Parameter(torch.randn((state_size,state_size)), requires_grad=True) # Temporal correlation and diffusion
        self.C = nn.Parameter(torch.randn((state_size, brownian_size)), requires_grad=True)  # Correlation matrix CC^T=\Sigma.
        self.noise_type = "additive"
        self.sde_type = "ito"
    
    def f(self, t, y):
        return y @ self.A.T
    
    def g(self, t, y):
        return self.C.repeat(y.size(0),1,1)

In [8]:
state_size: int = 10
brownian_size = 2
hidden: int = 64
lr: float = 1e-3
epochs: int = 30
batch_size: int = 32
method: str = "euler"  # or 'srk' / 'heun' / 'milstein' etc.
dt: float = 1e-2       # step for fixed-step solvers like euler/heun
device: str = "cuda" if torch.cuda.is_available() else "cpu"

In [9]:
dataset = VariableLengthSDEData(n_paths=800, length_range=(12, 64), theta=2.0, mu=0.0, sigma=0.5, device=device)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_variable, drop_last=False)

# Model
sde = LinearSDE(state_size=state_size, brownian_size=brownian_size).to(device)
ts_pad, y_pad, mask, x0 = next(iter(loader))
x_sim = torchsde.sdeint(sde, x0, ts_pad, method=method, dt=dt) 

In [10]:
def masked_mse(pred, target, mask):
    # pred, target: (B, T, d); mask: (B, T)
    diff = (pred - target) ** 2  # (B, T, d)
    diff = diff.mean(dim=-1)     # (B, T)
    diff = diff * mask           # zero out padded steps
    # average over valid elements only
    denom = mask.sum().clamp_min(1.0)
    return diff.sum() / denom

def train():
    set_seed(123)
    # Data
    dataset = VariableLengthSDEData(n_paths=800, length_range=(12, 64), theta=2.0, mu=0.0, sigma=0.5, device=device)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_variable, drop_last=False)

    # Model
    sde = LinearSDE(state_size=state_size, brownian_size=brownian_size).to(device)
    optimizer = optim.Adam(sde.parameters(), lr=lr)

    # Training loop
    for epoch in range(1, epochs + 1):
        sde.train()
        total_loss = 0.0
        n_batches = 0

        for ts_pad, y_pad, mask, x0 in loader:
            # ts_pad: (T_max,), y_pad: (B, T_max, d), mask: (B, T_max), x0: (B, d)
            ts_pad = ts_pad.to(device)
            y_pad = y_pad.to(device)
            mask = mask.to(device)
            x0 = x0.to(device)

            optimizer.zero_grad()

            # sdeint expects state shape (B, d); returns (T, B, d)
            # Keep noise consistent per-step across batch by default BrownianMotion.
            x_sim = torchsde.sdeint(sde, x0, ts_pad, method=method, dt=dt)  # (T_max, B, d)

            x_sim = x_sim.transpose(0, 1)  # (B, T_max, d)

            loss = masked_mse(x_sim, y_pad, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sde.parameters(), 5.0)
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        avg_loss = total_loss / max(1, n_batches)
        print(f"Epoch {epoch:03d} | train loss: {avg_loss:.6f}")

In [ ]:
train()

Epoch 001 | train loss: 1.071184
Epoch 002 | train loss: 0.971841
Epoch 003 | train loss: 0.929083
Epoch 004 | train loss: 0.910933
Epoch 005 | train loss: 0.863236
Epoch 006 | train loss: 0.846944
Epoch 007 | train loss: 0.743459
Epoch 008 | train loss: 0.740673
Epoch 009 | train loss: 0.688663
Epoch 010 | train loss: 0.644282
Epoch 011 | train loss: 0.645606
Epoch 012 | train loss: 0.633634
Epoch 013 | train loss: 0.570145
Epoch 014 | train loss: 0.571088
Epoch 015 | train loss: 0.539405
Epoch 016 | train loss: 0.529271
Epoch 017 | train loss: 0.526551
Epoch 018 | train loss: 0.486004
Epoch 019 | train loss: 0.467384
Epoch 020 | train loss: 0.458615
Epoch 021 | train loss: 0.438762
Epoch 022 | train loss: 0.434501
Epoch 023 | train loss: 0.424833
